<a href="https://colab.research.google.com/github/SravyaKodati/Activity-of-Daily-Living/blob/dev/hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [2]:
!pip install fiftyone

In [3]:
!pip install datasets

In [4]:
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

In [7]:
from datasets import load_dataset

# Loading the dataset

In [8]:
dataset = load_dataset("Voxel51/GMNCSA24-FO", streaming= True) #not using this

README.md:   0%|          | 0.00/3.15k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/162 [00:00<?, ?it/s]

In [11]:
d = load_from_hub("Voxel51/GMNCSA24-FO", overwrite= True)

INFO:fiftyone.utils.huggingface:Downloading config file fiftyone.yml from Voxel51/GMNCSA24-FO


Loading dataset


INFO:fiftyone.utils.huggingface:Loading dataset


Importing samples...


INFO:fiftyone.utils.data.importers:Importing samples...


...

INFO:eta.core.utils: 100% |█████████████████| 335/335 [27.6ms elapsed, 0s remaining, 12.1K samples/s]     


Importing frames...


INFO:fiftyone.utils.data.importers:Importing frames...


 100% |█████████████████████| 0/0 [6.4ms elapsed, ? remaining, ? samples/s]  


INFO:eta.core.utils: 100% |█████████████████████| 0/0 [6.4ms elapsed, ? remaining, ? samples/s]  


In [12]:
print(d)

Name:        Voxel51/GMNCSA24-FO
Media type:  video
Num samples: 335
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.VideoMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    sample_id:        fiftyone.core.fields.ObjectIdField
    support:          fiftyone.core.fields.FrameSupportField
    events:           fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
Frame fields:
    id:               fiftyone.core.fields.ObjectIdField
    frame_number:     fiftyone.core.fields.FrameNumberField
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField


In [13]:
video_paths = d.values("filepath")
labels = d.values("events.label")
print(video_paths[:5])
print(set(labels))

['/root/fiftyone/huggingface/hub/Voxel51/GMNCSA24-FO/data/01.mp4', '/root/fiftyone/huggingface/hub/Voxel51/GMNCSA24-FO/data/01.mp4', '/root/fiftyone/huggingface/hub/Voxel51/GMNCSA24-FO/data/01.mp4', '/root/fiftyone/huggingface/hub/Voxel51/GMNCSA24-FO/data/02.mp4', '/root/fiftyone/huggingface/hub/Voxel51/GMNCSA24-FO/data/02.mp4']
{'Walking', 'Standing', 'Eating', 'Falling (FW)', 'Falling (BW)', 'Writing', 'Sitting', 'Sleeping', 'Falling (BW', 'Drinking', 'Falling (SW)', 'Reading', 'Exercising', 'Fall (FW)'}


# Preprocessing

In [14]:
from datasets import Dataset

hf_dataset = Dataset.from_dict({
    "video_path": video_paths,
    "label": labels,
})
print(hf_dataset)

Dataset({
    features: ['video_path', 'label'],
    num_rows: 335
})


In [15]:
label_mapping = {
    "Falling (BW": "Falling (BW)",
    "Fall (FW)": "Falling (FW)",
}

In [16]:
video_paths = d.values("filepath")
labels = d.values("events.label")

cleaned_labels = [label_mapping.get(label, label) for label in labels]

In [17]:
hf_dataset = Dataset.from_dict({
    "video_path": video_paths,
    "label": cleaned_labels,
})

In [19]:
set(cleaned_labels)

{'Drinking',
 'Eating',
 'Exercising',
 'Falling (BW)',
 'Falling (FW)',
 'Falling (SW)',
 'Reading',
 'Sitting',
 'Sleeping',
 'Standing',
 'Walking',
 'Writing'}

In [18]:
print(len(set(cleaned_labels)))

12


In [34]:
split_dataset = hf_dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [20]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

GPU available: True
Number of GPUs: 1


In [22]:
import cv2
import torch
from transformers import VideoMAEImageProcessor

image_processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

def preprocess_video(example):
    """Loads video, extracts frames, and preprocesses them for VideoMAE."""
    video_path = example["video_path"]
    cap = cv2.VideoCapture(video_path)

    frames = []
    success, frame = cap.read()
    while success:
        frames.append(frame)
        success, frame = cap.read()

    cap.release()

    pixel_values = image_processor(frames, return_tensors="pt")["pixel_values"] # Convert frames to tensors

    return {"pixel_values": pixel_values, "label": example["label"]}

In [ ]:
hf_dataset = hf_dataset.map(preprocess_video)

In [36]:
sample_train= train_dataset.select(range(20))
sample_eval = eval_dataset.select(range(3))

In [37]:
sample_train = sample_train.map(preprocess_video)
sample_eval = sample_eval.map(preprocess_video)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [41]:
label2id = {label: idx for idx, label in enumerate(set(cleaned_labels))}
id2label = {idx: label for label, idx in label2id.items()}

# Model Checkpointing

In [29]:
from transformers import TrainingArguments, Trainer, VideoMAEForVideoClassification

model_ckpt = "MCG-NJU/videomae-base"
model = VideoMAEForVideoClassification.from_pretrained(
    model_ckpt,
    num_labels=len(set(labels)),
)

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
model = VideoMAEForVideoClassification.from_pretrained(
    model_ckpt,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sample_train,
    eval_dataset=sample_eval
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
trainer.train()